<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/05-limpieza-de-datos/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 0.5 · Limpieza de datos: vacíos, duplicados, textos y fechas

**Serie:** Del Excel al Machine Learning · **Temporada 0** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Abrir un CSV separado por punto y coma, y revisar su estado con `.info()`.
- Quitar filas **duplicadas** y unificar **textos** mal escritos.
- Decidir qué hacer con las celdas **vacías**.
- Convertir en números y fechas los datos guardados como **texto**.
- Encontrar datos **absurdos**.

## 📊 En Excel lo harías así

La oficina te pasa el archivo de ventas del sistema, y viene sucio. Entonces:

1. Usas **Datos → Quitar duplicados**.
2. Limpias textos con `=ESPACIOS()` (`TRIM`) y **Buscar y reemplazar**.
3. Le quitas el `S/` a los montos, para que Excel los sume.
4. Conviertes las fechas que quedaron como texto.

Y el próximo mes repites todo a mano. En Python lo escribes una vez, y luego solo vuelves a ejecutarlo.

## 🐍 En Python

### Paso 1: abrir y revisar

Empezamos igual que en el episodio 0.4:

In [1]:
import pandas as pd

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

Abrimos solo 3 filas con `nrows=3`, para mirar:

In [2]:
pd.read_csv(DATOS + "ventas_crudo.csv", nrows=3)

,Nro Venta;Fecha;Cliente;Producto;Vendedor;Ciudad;Cantidad;Precio Unitario;Descuento;Total;Método de Pago
0,V000001;02/01/2024;C0002;P022;E006;Arequipa;3;...
1,V000002;02/01/2024;C0003;P045;E004;Lima;2;S/ 4...
2,V000003;02/01/2024;C0008;P021;E010;Chiclayo;3;...


¡Todo quedó en una sola columna! Este archivo separa los datos con punto y coma, no con comas. Se lo indicamos con `sep=";"`. (Sin esa opción, el archivo completo da un `ParserError`).

> 💡 Si un archivo tuyo da `UnicodeDecodeError`, prueba agregando `encoding="latin-1"`.

De paso, ordenamos la fila de títulos (`ventas.columns`): en minúsculas, sin espacios y con los mismos nombres del archivo limpio. `.str` aplica funciones de texto a muchos valores a la vez, y `.rename()` recibe un diccionario, de nombre viejo a nombre nuevo. Quitamos la tilde para no tener que escribirla cada vez.

In [3]:
ventas = pd.read_csv(DATOS + "ventas_crudo.csv", sep=";")

ventas.columns = ventas.columns.str.lower().str.replace(" ", "_")
ventas = ventas.rename(
    columns={
        "nro_venta": "id_venta",
        "cliente": "id_cliente",
        "producto": "id_producto",
        "vendedor": "id_vendedor",
        "método_de_pago": "metodo_pago",
    }
)

ventas.head()

,id_venta,fecha,id_cliente,id_producto,id_vendedor,ciudad,cantidad,precio_unitario,descuento,total,metodo_pago
0,V000001,02/01/2024,C0002,P022,E006,Arequipa,3.0,S/ 48.00,0%,S/ 144.00,Efectivo
1,V000002,02/01/2024,C0003,P045,E004,Lima,2.0,S/ 46.00,0%,S/ 92.00,Billetera digital
2,V000003,02/01/2024,C0008,P021,E010,Chiclayo,3.0,S/ 46.00,0%,S/ 138.00,Efectivo
3,V000004,02/01/2024,C0022,P046,E007,Trujillo,9.0,S/ 70.00,0%,S/ 630.00,Efectivo
4,V000005,02/01/2024,C0024,P021,E013,Cusco,6.0,S/ 46.00,0%,S/ 276.00,Crédito


Ahora, el diagnóstico. `.info()` muestra cuántas celdas con datos tiene cada columna (*Non-Null Count*) y su tipo (*Dtype*). Fíjate en dos cosas: hay columnas con menos datos que filas, y `total` y `fecha` son texto (`object` o `str`, según la versión de pandas).

In [4]:
ventas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50750 entries, 0 to 50749
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_venta         50750 non-null  object 
 1   fecha            50750 non-null  object 
 2   id_cliente       50750 non-null  object 
 3   id_producto      50750 non-null  object 
 4   id_vendedor      50750 non-null  object 
 5   ciudad           50750 non-null  object 
 6   cantidad         50233 non-null  float64
 7   precio_unitario  50750 non-null  object 
 8   descuento        50750 non-null  object 
 9   total            50750 non-null  object 
 10  metodo_pago      49195 non-null  object 
dtypes: float64(1), object(10)
memory usage: 4.3+ MB


### Paso 2: quitar duplicados

`.duplicated()` marca las filas repetidas con `True`. Al sumarlas, cada `True` cuenta como 1. `.drop_duplicates()` las quita, igual que **Quitar duplicados** en Excel. Aquí las copias son idénticas; en tus archivos, vuelve a quitar duplicados después de unificar los textos, porque `lima` y `Lima` pueden esconder copias.

In [5]:
print(ventas.duplicated().sum())

ventas = ventas.drop_duplicates()

ventas.shape

750


(50000, 11)

### Paso 3: unificar textos

`.value_counts()` cuenta cuántas veces aparece cada valor, como una tabla dinámica rápida. Mira cuántas formas hay de escribir lo mismo. `Efectivo` sale dos veces porque uno tiene un espacio invisible al final:

In [6]:
ventas["metodo_pago"].value_counts()

metodo_pago
Efectivo             15031
Billetera digital    11561
Transferencia        10260
Crédito               7750
EFECTIVO               455
Efectivo               442
efectivo               407
yape                   359
Plin                   334
Yape                   319
TRANSFERENCIA          307
transferencia          304
Transferencia          287
Crédito                226
crédito                215
CRÉDITO                211
Name: count, dtype: int64

Lo arreglamos en tres pasos, y guardamos el resultado encima de la misma columna:

- `.str.strip()` quita los espacios de los extremos, parecido a `=ESPACIOS()` (`TRIM`).
- `.str.capitalize()` deja la primera letra en mayúscula y el resto en minúscula. Es parecido a `=NOMPROPIO()` (`PROPER`).
- `.replace()` es **Buscar y reemplazar**, con un diccionario, pero solo cambia celdas completas. Para cambiar una parte del texto se usa `.str.replace()`, como veremos con los montos.

In [7]:
ventas["metodo_pago"] = ventas["metodo_pago"].str.strip().str.capitalize()
ventas["metodo_pago"] = ventas["metodo_pago"].replace(
    {"Yape": "Billetera digital", "Plin": "Billetera digital"}
)

ventas["metodo_pago"].value_counts()

metodo_pago
Efectivo             16335
Billetera digital    12573
Transferencia        11158
Crédito               8402
Name: count, dtype: int64

### Paso 4: celdas vacías

`.isna()` marca las celdas vacías, y `.sum()` las cuenta por columna. Aquí no hay una regla única: **tú decides** según lo que signifique el dato. Si no sabemos cómo se pagó, no vamos a inventarlo ni a borrar la venta. Con `.fillna()` escribimos `Sin dato`:

In [8]:
print(ventas.isna().sum())

ventas["metodo_pago"] = ventas["metodo_pago"].fillna("Sin dato")

id_venta              0
fecha                 0
id_cliente            0
id_producto           0
id_vendedor           0
ciudad                0
cantidad            509
precio_unitario       0
descuento             0
total                 0
metodo_pago        1532
dtype: int64


Los vacíos de `cantidad` no los llenamos con 0, porque sería inventar una venta de cero. Se pueden recalcular, y lo harás en el ejercicio. Por esos vacíos, pandas la muestra con decimales (`3.0`).

### Paso 5: montos guardados como texto

`S/ 1,250.50` no se puede sumar. Le quitamos el símbolo y la coma, y lo convertimos en número con `.astype(float)`, como `=VALOR()` (`VALUE`) con toda una columna.

> ⚠️ Si vuelves a ejecutar esta celda, verás `Can only use .str accessor`. No está mal tu código: la columna ya es número. Vuelve a ejecutar desde la celda que abre el archivo.

In [9]:
ventas["total"] = ventas["total"].str.replace("S/ ", "").str.replace(",", "").astype(float)

print(round(ventas["total"].sum(), 2))

13582286.61


### Paso 6: fechas guardadas como texto

Aquí hay una trampa: ¿`02/01/2024` es 2 de enero o 1 de febrero? Si dejas que la computadora adivine, puede equivocarse sin avisarte. Por eso, **dile siempre el formato**.

Este archivo tiene casi todas las fechas como `día/mes/año`, y unas pocas como `año-mes-día`. Primero guardamos el texto original en `texto`, porque la siguiente línea reemplaza la columna. Con `errors="coerce"` (forzar), lo que no calza con el formato queda vacío, en vez de dar error. Luego `.fillna()` completa esos vacíos con el segundo formato, parecido a `=SI.ERROR()` (`IFERROR`):

In [10]:
texto = ventas["fecha"]

ventas["fecha"] = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
ventas["fecha"] = ventas["fecha"].fillna(pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce"))

print(ventas["fecha"].isna().sum())

0


Son `%d` para el día, `%m` para el mes y `%Y` para el año. Ahora sí son fechas: `.dt` saca sus partes, como `=AÑO()` (`YEAR`) y `=MES()` (`MONTH`):

In [11]:
ventas["fecha"].dt.year.head(2)

0    2024
1    2024
Name: fecha, dtype: int32

### Paso 7: datos absurdos

Ninguna fórmula avisa de un error de tipeo. Por eso, ordena y mira los extremos:

In [12]:
ventas.sort_values("cantidad", ascending=False)[["cantidad", "precio_unitario", "total"]].head(6)

,cantidad,precio_unitario,total
26527,600.0,S/ 116.00,696.0
48573,200.0,S/ 37.00,74.0
11061,200.0,S/ 56.00,112.0
27692,100.0,S/ 24.00,24.0
50,100.0,S/ 102.00,102.0
39984,41.0,S/ 40.00,1476.0


¿600 fardos por ese total? Son errores de tipeo: alguien escribió dos ceros de más. El `total` nos deja calcular la cantidad correcta. Lo harás en el ejercicio.

Para eso vas a operar con columnas enteras: `ventas["total"] / 100` divide fila por fila, como arrastrar una fórmula en Excel.

> ⚠️ **Nunca limpies a ciegas.** Antes de cambiar algo, cuenta cuántas filas afecta, y al final comprueba que los números cuadran.

## 🤖 Pídeselo a la IA

La limpieza es ideal para pedir ayuda. Escribe ejemplos **inventados, con el mismo formato**. No pegues datos de clientes ni de tu empresa:

> *"Tengo una columna de pandas con montos como `S/ 1,250.50`, guardados como texto. ¿Cómo los convierto en números? Explícame cada paso."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y sobre todo comprueba el resultado.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Quitar duplicados | `ventas.drop_duplicates()` |
| `=ESPACIOS()` (`TRIM`), aproximadamente | `.str.strip()` |
| `=NOMPROPIO()` (`PROPER`), aproximadamente | `.str.capitalize()` |
| Buscar y reemplazar una celda completa | `.replace()` |
| Buscar y reemplazar parte del texto | `.str.replace()` |
| Contar vacíos | `.isna().sum()` |
| Llenar vacíos | `.fillna("Sin dato")` |
| `=VALOR()` (`VALUE`) | `.astype(float)` |
| Convertir texto en fecha | `pd.to_datetime(texto, format="%d/%m/%Y")` |
| `=AÑO()` (`YEAR`), `=MES()` (`MONTH`) | `.dt.year`, `.dt.month` |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/05-limpieza-de-datos/ejercicio.ipynb) y termina de limpiar el archivo. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/05-limpieza-de-datos/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0